The below code is used to mount your google drive into this notebook, it should ask for your sign in and permission, it will then download the embedded files from a repository onto your drive
(check the output for this cell)

In [ ]:
from google.colab import drive
import os, requests

drive.mount('/content/drive/')

RAG_FOLDER = '/content/drive/MyDrive/RAGFiles'
GITHUB = 'https://raw.githubusercontent.com/Cocallian/SOPAIDict/main/PreProcessingSAHPRAGuidelines/PreProcessesV1/'
FILES = ['passages.jsonl', 'passage_vectors.npy', 'passage_ids.json']

answer = input("Create a 'RAGFiles' folder in your Google Drive and download the RAG files into it? (y/n): ")

if answer.strip().lower() == 'y':
    os.makedirs(RAG_FOLDER, exist_ok=True)
    for name in FILES:
        r = requests.get(GITHUB + name)
        r.raise_for_status()                      # stops with an error if the download fails
        with open(os.path.join(RAG_FOLDER, name), 'wb') as f:
            f.write(r.content)
        print("Downloaded", name)
    os.chdir(RAG_FOLDER)
else:
    print("Skipped. The RAG cell needs these files in", RAG_FOLDER)



Debug code below to find out where you are in your drive, ADD THE FILES  in PreProcessingSAHPRAGuidelines/PreProcessV1 to your GOOGLE DRIVE in a folder, this is so that google colab can access them, otherwise IT DOES NOT WORK

In [ ]:
import os

for folder, subfolders, files in os.walk("/content/drive/MyDrive/RAGFiles"): #change this text file to your folder path
    for name in files:
        print(os.path.join(folder, name))

The below code installs both ollama(the server architecture to host the llm) and the chosen model qwen3.5:0.8b

In [ ]:
import shutil, subprocess, time, requests

MODEL = "qwen3.5:0.8b"

# 1. Install Ollama if it isn't installed yet
if not shutil.which("ollama"):
    print("Installing Ollama...")
    subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null && "
                   "curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True)

# 2. Start the Ollama server and wait until it answers
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=5)
        print("Ollama is running")
        break
    except requests.exceptions.RequestException:   
        time.sleep(2)
else:
    raise RuntimeError("Ollama didn't start. Check the install step.")

# 3. Download Qwen (about 6 GB; skipped if already downloaded)
print("Downloading Qwen (a few minutes the first time)...")
pull = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if pull.returncode != 0:
    raise RuntimeError(pull.stderr[-500:])
print("Model ready")

The below code is the actual model and RAG pipeline working, change the text in the question variable to test a query here, later this colab notebook will be able to connect to the frontend php page

In [ ]:
import os, sys, json, warnings
from pathlib import Path

# keep Hugging Face warnings and progress bars out of the output, since PHP shows everything this script prints
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")

import numpy as np, requests
from sentence_transformers import SentenceTransformer

MODEL = "qwen3.5:0.8b"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"                 # must be the same model EmbedScript.py used
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # BGE wants this on questions only
TOP_K = 5                                               # passages given to Qwen; raise num_ctx if you raise this

#sys.stdout.reconfigure(encoding="utf-8")   # so PHP receives special characters correctly

question = "Explain the correct steps to dispose of radioactive waste"#sys.argv[1] if len(sys.argv) > 1 else "In one sentence, what does SAHPRA regulate?"

# Load the passages and the vectors EmbedScript.py saved for them
with open("passages.jsonl", encoding="utf-8") as f:
    passages = [json.loads(line) for line in f]
vectors = np.load("passage_vectors.npy")
if len(vectors) != len(passages):
    sys.exit("The passages have changed since they were embedded. Run EmbedScript.py again.")

# Embed the question and find the passages most similar to it
embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True)[0]
best = np.argsort(-(vectors @ q))[:TOP_K]


def source_label(p):
    pages = f"p. {p['page_start']}" if p["page_start"] == p["page_end"] else f"pp. {p['page_start']}-{p['page_end']}"
    return f"{p['document_number'] or p['source_file']}: {p['title']}, {pages}"


sources = "\n\n".join(f"[{i}] {source_label(passages[j])}\n{passages[j]['text']}" for i, j in enumerate(best, start=1))

SYSTEM_PROMPT = (
    "You answer questions about South African Health Products Regulatory Authority (SAHPRA) guidelines. "
    "Use ONLY the numbered sources provided. Cite sources in square brackets, e.g. [1] or [2][3]. "
    "If the sources do not contain the answer, say you could not find it in the guidelines."
)

try:
    reply = requests.post("http://localhost:11434/api/chat", json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {question}"},
        ],
        "stream": False,
        "think": False,
        "options": {"num_ctx": (4096), "temperature": 0.2},   # small context window = less memory; low temperature = sticks to the sources
    }, timeout=300)
except requests.ConnectionError:
    sys.exit("Ollama isn't running. Start the Ollama app and try again.")

data = reply.json()


print(data["message"]["content"])
print("\nSources:")
for i, j in enumerate(best, start=1):
    print(f"[{i}] {source_label(passages[j])}")
